# Group 10 – Dublin Bikes Availability Prediction Notebook

## Table of Contents
1. Project Goal
2. Load and Inspect the Dataset
3. Data Cleaning
4. Feature Engineering
5. Target Variable and Feature Selection
6. Train-Test Split
7. Model Training and Comparison
8. Final Model Selection
9. Save the Final Model
10. Conclusion and Web Integration Plan


## 1. Project Goal

The goal of this notebook is to develop a machine learning model to predict the number of available bikes at a selected Dublin Bikes station for a user-specified time.

This is a regression problem, because the target variable is a numeric value: the number of bikes available at a station.

The final web application is intended to use:
- user input (station, date, and time),
- station-related information,
- time-based features,
- and forecast weather data

to generate a prediction and display the result on the bike-sharing website.

The purpose of this notebook is to:
1. inspect and clean the dataset,
2. engineer and select practical features,
3. compare different regression models,
4. choose a final model for deployment,
5. and prepare the model for integration into the web application.

## 2. Load and Inspect the Dataset

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("final_merged_data.csv")

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))
print("First 15 columns:", df.columns[:15].tolist())
df.head()

Dataset shape: (298946, 78)
Number of columns: 78
First 15 columns: ['last_reported', 'station_id', 'num_bikes_available', 'num_docks_available', 'is_installed', 'is_renting', 'is_returning', 'name', 'address', 'lat', 'lon', 'capacity', 'stno', 'year', 'month']


,last_reported,station_id,num_bikes_available,num_docks_available,is_installed,is_renting,is_returning,name,address,lat,...,min_humidity_quality_indicator,min_relative_humidity_percent,humidity_std_quality_indicator,relative_humidity_std_deviation,max_pressure_quality_indicator,max_barometric_pressure_hpa,min_pressure_quality_indicator,min_barometric_pressure_hpa,pressure_std_quality_indicator,barometric_pressure_std_deviation
0,2024-12-01 00:10:00,10,15,1,True,True,True,DAME STREET,Dame Street,53.344006,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
1,2024-12-01 00:10:00,100,17,8,True,True,True,HEUSTON BRIDGE (SOUTH),Heuston Bridge (South),53.347107,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
2,2024-12-01 00:10:00,109,20,9,True,True,True,BUCKINGHAM STREET LOWER,Buckingham Street Lower,53.353333,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
3,2024-12-01 00:10:00,11,1,29,True,True,True,EARLSFORT TERRACE,Earlsfort Terrace,53.334293,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083
4,2024-12-01 00:10:00,114,4,36,True,True,True,WILTON TERRACE (PARK),Wilton Terrace (Park),53.333652,...,0,83.2,0,0.284,0,1002.56,0,1002.26,0,0.083


The dataset contains 298,946 rows and 78 columns. It includes bike availability information, station-related attributes, time variables, and multiple weather measurements. Since not all variables are equally useful for deployment, the next steps will focus on inspection, cleaning, and selecting practical features for prediction.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 298946 entries, 0 to 298945
Data columns (total 78 columns):
 #   Column                                  Non-Null Count   Dtype  
---  ------                                  --------------   -----  
 0   last_reported                           298946 non-null  object 
 1   station_id                              298946 non-null  int64  
 2   num_bikes_available                     298946 non-null  int64  
 3   num_docks_available                     298946 non-null  int64  
 4   is_installed                            298946 non-null  bool   
 5   is_renting                              298946 non-null  bool   
 6   is_returning                            298946 non-null  bool   
 7   name                                    298946 non-null  object 
 8   address                                 298946 non-null  object 
 9   lat                                     298946 non-null  float64
 10  lon                                     2989

In [4]:
missing_counts = df.isnull().sum()
missing_counts[missing_counts > 0].sort_values(ascending=False)

Series([], dtype: int64)

## 3. Data Cleaning

The dataset has no missing values. In this step, basic cleaning checks are performed to remove duplicate rows and inspect whether the target and key numeric variables fall within valid ranges.

In [5]:
print("Duplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [6]:
df = df.drop_duplicates().copy()
print("Shape after dropping duplicates:", df.shape)

Shape after dropping duplicates: (298946, 78)


In [7]:
print("Negative available bikes:", (df["num_bikes_available"] < 0).sum())
print("Negative available docks:", (df["num_docks_available"] < 0).sum())
print("Negative capacity:", (df["capacity"] < 0).sum())
print("Bikes greater than capacity:", (df["num_bikes_available"] > df["capacity"]).sum())
print("Docks greater than capacity:", (df["num_docks_available"] > df["capacity"]).sum())
print("Bikes + docks not equal to capacity:", ((df["num_bikes_available"] + df["num_docks_available"]) != df["capacity"]).sum())

Negative available bikes: 0
Negative available docks: 0
Negative capacity: 0
Bikes greater than capacity: 0
Docks greater than capacity: 0
Bikes + docks not equal to capacity: 15555


The dataset contains no duplicate rows and no negative values for key bike availability variables. A subset of records shows that the sum of available bikes and available docks does not exactly match station capacity. This may reflect operational or temporary status differences in real-world bike-sharing data, so these records are retained rather than removed.

## 4. Feature Engineering

To support prediction, additional time-based features are created from the timestamp information. These features are intended to capture daily and weekly usage patterns that may influence bike availability.

In [8]:
df["last_reported"] = pd.to_datetime(df["last_reported"])

In [9]:
df["day_of_week"] = df["last_reported"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["rush_hour"] = df["hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)

df[["last_reported", "hour", "day_of_week", "is_weekend", "rush_hour"]].head()

,last_reported,hour,day_of_week,is_weekend,rush_hour
0,2024-12-01 00:10:00,0,6,1,0
1,2024-12-01 00:10:00,0,6,1,0
2,2024-12-01 00:10:00,0,6,1,0
3,2024-12-01 00:10:00,0,6,1,0
4,2024-12-01 00:10:00,0,6,1,0


In [10]:
print(df[["day_of_week", "is_weekend", "rush_hour"]].describe())

         day_of_week     is_weekend      rush_hour
count  298946.000000  298946.000000  298946.000000
mean        2.923478       0.282205       0.285396
std         2.060851       0.450073       0.451604
min         0.000000       0.000000       0.000000
25%         1.000000       0.000000       0.000000
50%         3.000000       0.000000       0.000000
75%         5.000000       1.000000       1.000000
max         6.000000       1.000000       1.000000


## 5. Target Variable and Feature Selection

The target variable for this project is `num_bikes_available`, which represents the number of bikes available at a station at a given time.

Feature selection is guided by two criteria:
1. predictive usefulness,
2. deployment practicality.

The final web application must be able to reconstruct the selected features at runtime using:
- user input (station, date, time),
- station information,
- and forecast weather data.

Therefore, the selected features should not only be informative, but also realistically available when the prediction is requested by the user.

In [11]:
candidate_features = [
    "station_id",
    "capacity",
    "hour",
    "month",
    "day_of_week",
    "is_weekend",
    "rush_hour",
    "lat",
    "lon",
    "max_air_temperature_celsius",
    "max_relative_humidity_percent",
    "max_barometric_pressure_hpa"
]

target = "num_bikes_available"

print("Selected target:", target)
print("Number of candidate features:", len(candidate_features))
print("Candidate features:", candidate_features)

Selected target: num_bikes_available
Number of candidate features: 12
Candidate features: ['station_id', 'capacity', 'hour', 'month', 'day_of_week', 'is_weekend', 'rush_hour', 'lat', 'lon', 'max_air_temperature_celsius', 'max_relative_humidity_percent', 'max_barometric_pressure_hpa']


The selected features combine three types of information:

- **Station-related features**: `station_id`, `capacity`, `lat`, and `lon` help represent station identity, size, and location.
- **Time-based features**: `hour`, `month`, `day_of_week`, `is_weekend`, and `rush_hour` help capture recurring demand patterns over time.
- **Weather-related features**: `max_air_temperature_celsius`, `max_relative_humidity_percent`, and `max_barometric_pressure_hpa` are included because weather conditions may influence bike usage.

These features were chosen because they are both meaningful for prediction and practical to reconstruct during deployment.

In [12]:
model_df = df[candidate_features + [target]].copy()

print("Model dataset shape:", model_df.shape)
model_df.head()

Model dataset shape: (298946, 13)


,station_id,capacity,hour,month,day_of_week,is_weekend,rush_hour,lat,lon,max_air_temperature_celsius,max_relative_humidity_percent,max_barometric_pressure_hpa,num_bikes_available
0,10,16,0,12,6,1,0,53.344006,-6.266802,14.01,84.3,1002.56,15
1,100,25,0,12,6,1,0,53.347107,-6.292041,14.01,84.3,1002.56,17
2,109,29,0,12,6,1,0,53.353333,-6.249319,14.01,84.3,1002.56,20
3,11,30,0,12,6,1,0,53.334293,-6.258503,14.01,84.3,1002.56,1
4,114,40,0,12,6,1,0,53.333652,-6.248345,14.01,84.3,1002.56,4


In [13]:
print(model_df.isnull().sum().sum())

0


## 6. Train-Test Split

The dataset is split into training and test sets so that model performance can be evaluated on unseen data. The training set is used to fit the models, while the test set is used to compare their predictive performance.

In [15]:
from sklearn.model_selection import train_test_split

X = model_df[candidate_features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (239156, 12)
X_test shape: (59790, 12)
y_train shape: (239156,)
y_test shape: (59790,)


In [16]:
X_train.head()

,station_id,capacity,hour,month,day_of_week,is_weekend,rush_hour,lat,lon,max_air_temperature_celsius,max_relative_humidity_percent,max_barometric_pressure_hpa
297711,117,40,18,12,1,0,1,53.343655,-6.231755,9.680,85.80,998.67
98076,9,24,1,12,2,0,0,53.343033,-6.263578,6.227,69.63,1030.29
134437,106,40,14,12,5,1,0,53.358930,-6.280337,7.043,86.30,1018.31
234026,103,40,18,12,1,0,1,53.354664,-6.278681,11.920,91.00,1019.59
115867,5,40,17,12,3,0,1,53.330660,-6.260177,5.196,82.40,1024.77


## 7. Model Training and Comparison

Four regression models are trained and compared in this notebook:

1. Linear Regression
2. Ridge Regression
3. Decision Tree Regressor
4. Random Forest Regressor

These models were chosen to provide a mix of baseline linear methods and tree-based methods. Performance is evaluated using MAE, RMSE, and R².

In [17]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [18]:
def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [19]:
results = []

results.append(
    evaluate_model(
        "Linear Regression",
        LinearRegression(),
        X_train, X_test, y_train, y_test
    )
)

results.append(
    evaluate_model(
        "Ridge Regression",
        Ridge(alpha=1.0),
        X_train, X_test, y_train, y_test
    )
)

results.append(
    evaluate_model(
        "Decision Tree",
        DecisionTreeRegressor(random_state=42, max_depth=12),
        X_train, X_test, y_train, y_test
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=12,
            random_state=42,
            n_jobs=-1
        ),
        X_train, X_test, y_train, y_test
    )
)

In [21]:
results_df_rounded = results_df.copy()
results_df_rounded["MAE"] = results_df_rounded["MAE"].round(3)
results_df_rounded["RMSE"] = results_df_rounded["RMSE"].round(3)
results_df_rounded["R2"] = results_df_rounded["R2"].round(4)
results_df_rounded

,Model,MAE,RMSE,R2
3,Random Forest,3.966,5.350,0.6976
2,Decision Tree,4.242,5.865,0.6366
0,Linear Regression,7.824,9.353,0.0756
1,Ridge Regression,7.826,9.353,0.0755


## 8. Final Model Selection

Among the four tested models, Random Forest achieved the best overall performance, with the lowest MAE and RMSE and the highest R² score.

Although model evaluation metrics are important, deployment practicality is also a key consideration in this project. The final website prediction feature must construct model inputs at runtime using:
- user-selected station and time,
- station-related information,
- and forecast weather data.

Random Forest was selected as the final model because it provides strong predictive performance while remaining practical to integrate into the web application.

In [22]:
best_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",12
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [23]:
best_preds = best_model.predict(X_test)

best_mae = mean_absolute_error(y_test, best_preds)
best_rmse = mean_squared_error(y_test, best_preds) ** 0.5
best_r2 = r2_score(y_test, best_preds)

print("Best Model: Random Forest")
print("MAE:", round(best_mae, 3))
print("RMSE:", round(best_rmse, 3))
print("R2:", round(best_r2, 4))

Best Model: Random Forest
MAE: 3.966
RMSE: 5.35
R2: 0.6976


## 9. Save the Final Model

The selected Random Forest model is saved so that it can later be loaded by the Flask backend and used in the bike-sharing website prediction feature.

In [25]:
import joblib

joblib.dump(best_model, "best_bike_model.pkl")
print("Model saved as best_bike_model.pkl")

Model saved as best_bike_model.pkl


In [26]:
loaded_model = joblib.load("best_bike_model.pkl")
sample_pred = loaded_model.predict(X_test.iloc[:5])
print(sample_pred)

[17.53141666 15.10418022  5.58721551  0.59064711  3.21365325]


## 10. Conclusion and Web Integration Plan

This notebook developed a regression-based machine learning pipeline to predict the number of available bikes at a selected Dublin Bikes station and time.

The process included:
- inspecting the dataset,
- performing basic cleaning checks,
- engineering time-based features,
- selecting practical deployment-oriented features,
- comparing four regression models,
- and choosing Random Forest as the final model.

Random Forest was selected because it achieved the strongest overall performance while also remaining practical for deployment in the web application.

In the final website workflow, the prediction feature will use:
- user-selected station,
- user-selected date and time,
- station-related information,
- and forecast weather data

to construct the model input and return the predicted number of available bikes to the frontend.

The saved `.pkl` model file can be loaded by the Flask backend and used as part of the website prediction route.

In [27]:
loaded_model = joblib.load("best_bike_model.pkl")
sample_pred = loaded_model.predict(X_test.iloc[:5])
print("Sample predictions:", sample_pred)

Sample predictions: [17.53141666 15.10418022  5.58721551  0.59064711  3.21365325]


### Practical Deployment Note

Although several models were compared, the final model was not selected based only on evaluation metrics. The deployment requirement of the bike-sharing website was also considered. The selected model must work with features that can realistically be reconstructed at runtime from user input, station information, and forecast weather data. Random Forest provided the best balance between predictive performance and practical deployment.